# 01 · Exploración de los datos

**Objetivo:** descargar los precios, limpiarlos y entender sus rendimientos antes de construir nada encima.

**Necesitas:** TODO 1.1, 1.2 y 1.3 de `src/data.py`. Checkpoint: `python -m pytest tests/test_data.py`.

**Teoría:** `docs/01_Datos_y_rendimientos.pdf`.

**Cómo usar este notebook.** Ejecuta las celdas en orden. Una celda que llama a una función que todavía es un TODO falla con `NotImplementedError: TODO x.y`: completa ese TODO, pasa su checkpoint de `pytest` y vuelve a ejecutarla (con `autoreload` no hace falta reiniciar el kernel).

Las celdas marcadas **`# TODO (análisis)`** son tuyas y no las corrige ningún test: sirven para mirar los datos y responder a las preguntas. Hasta que las completes no hacen nada (`...`).

In [ ]:
# Configuración: localiza la raíz del repo y recarga src/ al guardar cambios (no hace falta reiniciar)
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import backtest, config, data, metrics, plotting, strategies, walkforward

plotting.apply_style()
pd.set_option("display.width", 140)

## 1. Descarga

Si todavía no has descargado los datos, esta celda lo hace (equivale a `python -m src.data` en la terminal). Los precios se guardan *en bruto* en `data/raw/prices.parquet`; la limpieza se aplica al cargarlos.

In [ ]:
if not config.PRICES_FILE.exists():
    data.main([])

meta = json.loads(config.PRICES_META_FILE.read_text(encoding="utf-8"))
{k: meta[k] for k in ("rows", "first_date", "last_date", "downloaded_at", "yfinance")}

## 2. Precios en bruto

Antes de limpiar, mira qué ha llegado: huecos, fechas, valores raros.

In [ ]:
raw = data.load_prices(clean=False)
raw.tail()

In [ ]:
print("Fechas ordenadas:", raw.index.is_monotonic_increasing, "| Fechas repetidas:", raw.index.has_duplicates)
print("Precios <= 0:", int((raw <= 0).sum().sum()))
raw.isna().sum().rename("NaN").to_frame().T

In [ ]:
# Un gráfico por activo (10 series en un solo gráfico no se leen)
normalized = raw.apply(lambda s: s / s.dropna().iloc[0])
fig, axes = plt.subplots(2, 5, figsize=(15, 5.5), sharex=True)
for ax, ticker in zip(axes.flat, raw.columns):
    ax.plot(normalized.index, normalized[ticker], color=plotting.PALETTE[0], linewidth=1.2)
    ax.set_yscale("log")
    ax.set_title(ticker, fontsize=10)
    ax.tick_params(axis="x", labelrotation=45, labelsize=8)
fig.suptitle("Precio ajustado con base 1 al inicio (escala log)", x=0.01, ha="left")
fig.tight_layout()

> **Pregunta.** Son precios *ajustados* (`auto_adjust=True`: dividendos y splits). AAPL hizo un split 4:1 el 31-08-2020: ¿ves algún salto ese día? ¿Qué pasaría en un backtest con precios sin ajustar? ¿Y si una acción paga un dividendo del 4 %?

## 3. Limpieza (TODO 1.3)

In [ ]:
prices = data.clean_prices(raw)
print(f"{len(raw)} filas en bruto -> {len(prices)} limpias. NaN restantes: {int(prices.isna().sum().sum())}")
prices.head()

Los datos de Yahoo de estos tickers vienen bastante limpios, así que comprueba la limpieza con una tabla sucia hecha a mano. Recorre los 5 pasos del docstring de `clean_prices` y verifica cada uno en la salida.

In [ ]:
dirty = pd.DataFrame(
    {"A": [10.0, np.nan, -1.0, 11.0, 12.0, 12.5], "B": [np.nan, 5.0, 5.5, 5.4, 5.6, 5.7]},
    index=pd.to_datetime(["2024-01-03", "2024-01-01", "2024-01-02", "2024-01-04", "2024-01-04", "2024-01-05"]),
)
display(dirty)
data.clean_prices(dirty)

> **Preguntas.** ¿Por qué la fila del 2024-01-01 desaparece? ¿Qué valor queda el 2024-01-04 y por qué? ¿Por qué está prohibido el `bfill`?

## 4. Rendimientos (TODO 1.1 y 1.2)

In [ ]:
rets = data.simple_returns(prices)
logrets = data.log_returns(prices)
rets.describe().T.style.format("{:.4f}")

In [ ]:
# TODO (análisis): tabla por activo con
#   - rendimiento medio anualizado  (media diaria × 252)
#   - volatilidad anualizada        (desviación diaria × √252)
#   - su cociente (un "Sharpe" sin costes ni tipo libre)
# ordenada por el cociente de mayor a menor.
# Pista: pd.DataFrame({...}) con tres Series que comparten índice; sort_values(...).
by_asset = ...
by_asset

In [ ]:
# TODO (análisis): rendimientos simples frente a logarítmicos en el SPY.
#   1) ¿Cuál es la diferencia media y la máxima entre ambos?
#   2) ¿Qué día difieren más? (pista: idxmax). ¿Qué pasó ese día?
#   3) Comprueba que la suma de los log-rendimientos de cada activo es ln(P_final / P_inicial).
...

In [ ]:
# Distribución de los rendimientos diarios del SPY frente a una normal con la misma media y σ
spy = rets["SPY"].dropna()
grid = np.linspace(spy.min(), spy.max(), 400)
normal = np.exp(-0.5 * ((grid - spy.mean()) / spy.std()) ** 2) / (spy.std() * np.sqrt(2 * np.pi))
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(spy, bins=150, density=True, color=plotting.PALETTE[0], label="SPY")
ax.plot(grid, normal, color=plotting.PALETTE[1], label="normal (misma media y σ)")
ax.set_yscale("log")
ax.set_ylim(1e-2, None)
ax.set_title("Rendimientos diarios del SPY: colas gruesas")
ax.legend()
print(f"Curtosis en exceso: {spy.kurt():.1f} (0 en una normal). Peor día: {spy.min():.1%} el {spy.idxmin():%Y-%m-%d}")

> **Pregunta.** Con una normal, un día de −12 % sería un suceso de más de 10σ: no debería ocurrir en la vida del universo. ¿Qué implica esto para métricas que usan σ, como el Sharpe?

## 5. Correlaciones

In [ ]:
corr = rets.corr()
plotting.plot_sharpe_heatmap(corr, title="Correlación de los rendimientos diarios");

In [ ]:
# Correlación móvil de 1 año entre acciones (SPY) y bonos largos (TLT)
rolling_corr = rets["SPY"].rolling(config.TRADING_DAYS).corr(rets["TLT"])
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(rolling_corr.index, rolling_corr, color=plotting.PALETTE[0])
ax.axhline(0, color=plotting.AXIS, linewidth=0.8)
ax.set_title("Correlación SPY–TLT, ventana móvil de 252 días");

## 6. Preguntas de reflexión

1. **Sesgo de supervivencia.** Este universo se ha elegido *hoy*, sabiendo qué empresas siguen siendo grandes. ¿En qué dirección sesga los resultados del backtest? ¿Afecta igual a los ETF que a AAPL o MSFT?
2. **Simples o logarítmicos.** El motor usa rendimientos simples. ¿Por qué para agregar *entre activos* (una cartera) sirven los simples y no los log? ¿Y para agregar *en el tiempo*?
3. **La última fila.** Si descargas con el mercado abierto, la última fila puede ser un precio intradía. ¿Qué riesgo tiene para una señal calculada "al cierre"?
4. **Diversificación.** Mirando la matriz de correlaciones, ¿qué activos aportan más diversificación a una cartera de acciones? ¿Es estable esa relación en el tiempo?